# MNIST MLP3 — AdamW baseline

This is a **clean optimizer baseline**. It contains no trace-log projection,
adaptive ECS correction, WW-PGD retraction, or spectral-flow intervention.

The notebook runs **three independent complete training seeds**. Every
plotted mean is accompanied by a two-sided 95% Student-t confidence interval,

$$
\bar{x} \pm t_{0.975,\,n-1}\frac{s}{\sqrt{n}},
\qquad n=3.
$$

Epoch zero and every training epoch record full train/test cross-entropy and
accuracy plus the original WeightWatcher full-$M$ quantities. The original
midpoint is

$$
m_{\mathrm{mid}}
=
\left\lfloor
\frac{m_{\mathrm{detX}}+m_{\mathrm{PL}}}{2}
\right\rfloor .
$$

Optimizer definition: `torch.optim.AdamW` with learning rate $10^{-3}$, betas $(0.9,0.999)$, epsilon $10^{-8}$, and decoupled weight decay $10^{-2}$.

## Persistent output contract

All three baseline notebooks use the same `RUN_ROOT`. By default it is
`baseline/runs/` inside the current clone, which is `/tmp/...` when the clone
itself is in `/tmp`. Set `RG_BASELINE_RUN_ROOT` to use another shared local
directory. This notebook writes aggregate CSVs and plots plus, for **every
seed**, `final_state.pt` and one complete model/optimizer checkpoint after
every epoch under `checkpoints/epoch_###.pt`.

Run all three baseline notebooks first, then run
`MNIST_MLP3_Baseline_Comparison.ipynb`; the comparison reads only these saved
artifacts and does not depend on live Python variables from another notebook.

In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

try:
    importlib.import_module("weightwatcher")
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "weightwatcher>=0.7.7"]
    )

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / "baseline"
    if (candidate / "rg_baselines").is_dir():
        ROOT = candidate
        break
    if (path / "rg_baselines").is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError(
        "Run this notebook from a clone of CalculatedContent/rg_optimizers."
    )
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def resolve_artifact_dir(environment_variable: str, default: Path) -> Path:
    raw = os.environ.get(environment_variable)
    path = Path(raw).expanduser() if raw else default
    if not path.is_absolute():
        path = Path.cwd() / path
    return path.resolve()


# All three training notebooks and the comparison notebook use this same root.
# Override it, for example, with:
#   export RG_BASELINE_RUN_ROOT=/tmp/rg_optimizers_baseline_runs
RUN_ROOT = resolve_artifact_dir("RG_BASELINE_RUN_ROOT", ROOT / "runs")
DATA_DIR = resolve_artifact_dir("RG_BASELINE_DATA_DIR", ROOT / "data")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("baseline root:", ROOT)
print("shared run root:", RUN_ROOT)
print("MNIST data directory:", DATA_DIR)

In [ ]:
from dataclasses import asdict
from IPython.display import display
import pandas as pd

from rg_baselines import (
    BaselineConfig,
    DEFAULT_BASELINE_SEEDS,
    plot_all_replicates,
    run_baseline_replicates,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 1000)

SEEDS = DEFAULT_BASELINE_SEEDS
assert len(SEEDS) == 3 and len(set(SEEDS)) == 3

CONFIG = BaselineConfig(
    optimizer="adamw",
    epochs=20,
    train_eval_max_batches=None,
    strict_metrics=True,
    adamw_learning_rate=1e-3,
    adamw_beta1=0.9,
    adamw_beta2=0.999,
    adamw_eps=1e-8,
    adamw_weight_decay=1e-2,
    save_epoch_checkpoints=True,
)
RUN_DIR = RUN_ROOT / CONFIG.run_slug
PLOT_DIR = RUN_DIR / "plots"

print("Independent seeds:", SEEDS)
print("run directory:", RUN_DIR)
print("epoch checkpoints enabled:", CONFIG.save_epoch_checkpoints)
display(pd.DataFrame([asdict(CONFIG)]))

In [ ]:
suite = run_baseline_replicates(
    CONFIG,
    seeds=SEEDS,
    data_dir=DATA_DIR,
    output_dir=RUN_DIR,
    progress=True,
    confidence=0.95,
)

plot_all_replicates(
    suite,
    output_dir=PLOT_DIR,
    show=True,
)

# Fail fast if any result needed by the comparison notebook was not persisted.
expected_paths = [
    RUN_DIR / "performance_by_epoch_and_seed.csv",
    RUN_DIR / "spectral_metrics_by_epoch_layer_and_seed.csv",
    RUN_DIR / "performance_summary_95ci.csv",
    RUN_DIR / "spectral_summary_95ci.csv",
    RUN_DIR / "replicate_manifest.json",
]
for seed in SEEDS:
    seed_dir = RUN_DIR / "seeds" / f"seed_{seed}"
    expected_paths.extend(
        [
            seed_dir / "performance_by_epoch.csv",
            seed_dir / "spectral_metrics_by_epoch_and_layer.csv",
            seed_dir / "esd_history.npz",
            seed_dir / "config.json",
            seed_dir / "final_state.pt",
        ]
    )
    expected_paths.extend(
        seed_dir / "checkpoints" / f"epoch_{epoch:03d}.pt"
        for epoch in range(1, CONFIG.epochs + 1)
    )

missing = [path for path in expected_paths if not path.is_file()]
if missing:
    raise RuntimeError(
        "The baseline run completed but required persisted artifacts are missing:\n"
        + "\n".join(f"  - {path}" for path in missing)
    )

print("saved aggregate results:", RUN_DIR)
print("saved plots:", PLOT_DIR)
print("verified persisted files:", len(expected_paths))

## Performance per epoch

The table below reports the mean, sample standard deviation, standard error,
and two-sided 95% Student-t confidence interval across the three complete
runs. `test_accuracy` is shown separately first so the primary outcome
cannot be overlooked. Classification perplexity is derived as
$\exp(\mathrm{cross\ entropy})$ and is also persisted in the comparison
notebook.

In [ ]:
performance_columns = [
    "epoch", "metric", "n", "mean", "std", "sem",
    "ci_half_width", "ci_low", "ci_high", "minimum", "maximum",
]
test_accuracy_summary = suite.performance_summary.loc[
    suite.performance_summary["metric"].eq("test_accuracy"),
    performance_columns,
].sort_values("epoch")
display(test_accuracy_summary)

required_performance = suite.performance_summary.loc[
    suite.performance_summary["metric"].isin(
        ["train_loss", "test_loss", "train_accuracy", "test_accuracy"]
    ),
    performance_columns,
].sort_values(["metric", "epoch"])
display(required_performance)

## WeightWatcher and midpoint metrics per epoch

These rows aggregate the original WeightWatcher outputs across seeds.
`detX_num`, `num_pl_spikes`, and `ERG_gap` come directly from
`watcher.analyze(ERG=True)`. The midpoint and trace-log coordinates are then
computed from the WeightWatcher-rescaled ESD.

In [ ]:
required_ww_metrics = [
    "alpha",
    "detX_num",
    "num_pl_spikes",
    "ERG_gap",
    "m_midpoint",
    "trace_log_midpoint_per_eval",
    "trace_log_midpoint_total",
]
spectral_columns = [
    "layer", "epoch", "metric", "n", "mean", "std", "sem",
    "ci_half_width", "ci_low", "ci_high", "minimum", "maximum",
]
required_spectral = suite.spectral_summary.loc[
    suite.spectral_summary["metric"].isin(required_ww_metrics),
    spectral_columns,
].sort_values(["metric", "layer", "epoch"])
display(required_spectral)

## Additional per-epoch diagnostics

The following table includes effective ranks, retained-energy fractions,
spectral conditioning, normalization audits, gradient norms, parameter
norms, and timing. All aggregate rows retain the same seed-level 95%
confidence-interval definition.

In [ ]:
additional_spectral_metrics = [
    "stable_rank",
    "participation_ratio",
    "entropy_effective_rank",
    "boundary_overlap_ratio",
    "top1_energy_fraction",
    "pl_energy_fraction",
    "detx_energy_fraction",
    "midpoint_energy_fraction",
    "geometric_mean_midpoint",
    "normalized_lambda_max",
    "normalized_lambda_midpoint_cut",
    "eigenvalue_condition_number",
]
display(
    suite.spectral_summary.loc[
        suite.spectral_summary["metric"].isin(additional_spectral_metrics),
        spectral_columns,
    ].sort_values(["metric", "layer", "epoch"])
)

display(
    suite.performance_summary.loc[
        suite.performance_summary["metric"].isin(
            [
                "mean_gradient_norm_before_clip",
                "max_gradient_norm_before_clip",
                "parameter_l2_norm",
                "train_time_sec",
                "evaluation_time_sec",
                "weightwatcher_time_sec",
            ]
        ),
        performance_columns,
    ].sort_values(["metric", "epoch"])
)

In [ ]:
# Final scientific and persistence audit.
expected_epochs = set(range(CONFIG.epochs + 1))
assert set(suite.performance["epoch"].astype(int)) == expected_epochs
assert set(suite.performance["seed"].astype(int)) == set(SEEDS)

valid = suite.spectral_metrics.loc[suite.spectral_metrics["status"].eq("ok")]
for epoch in expected_epochs:
    for layer in ("fc1", "fc2", "fc3"):
        observed = set(
            valid.loc[
                valid["epoch"].eq(epoch) & valid["layer"].eq(layer),
                "seed",
            ].astype(int)
        )
        assert observed == set(SEEDS), (epoch, layer, observed)

for seed in SEEDS:
    checkpoint_dir = RUN_DIR / "seeds" / f"seed_{seed}" / "checkpoints"
    saved_epochs = {
        int(path.stem.split("_")[-1])
        for path in checkpoint_dir.glob("epoch_*.pt")
    }
    assert saved_epochs == set(range(1, CONFIG.epochs + 1)), (
        seed,
        sorted(saved_epochs),
    )

print(
    "Audit passed:",
    len(SEEDS),
    "independent seeds;",
    CONFIG.epochs + 1,
    "metric checkpoints including epoch 0;",
    CONFIG.epochs,
    "model/optimizer epoch checkpoints per seed;",
    "FC1/FC2/FC3 WeightWatcher metrics at every epoch.",
)